In [161]:
import joblib
import pickle
import pandas as pd
import numpy as np



In [162]:
pkl_file_path = '/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/features/I3D_features.pkl'

features_df = pd.read_pickle(pkl_file_path)
print(features_df.head())
print(type(features_df), getattr(features_df, "shape", None))
print(features_df.columns)



              id                                       I3D_features
0    ZcVzjZeVwj0  [[[[[0.03112409]], [[0.00099271]], [[0.000154]...
1    ZIKWlQUff9c  [[[[[0.22031154]], [[0.22767238]], [[0.1475680...
2  FnTehW6ik-Y_2  [[[[[0.20237736]], [[0.38866925]], [[0.4987630...
3    0lrX7s_3ScY  [[[[[0.04070692]], [[0.00172393]], [[0.0040934...
4    bgwSjzhyPs8  [[[[[0.]], [[0.]], [[0.]], [[0.]], [[0.0005370...
<class 'pandas.core.frame.DataFrame'> (7050, 2)
Index(['id', 'I3D_features'], dtype='object')


In [8]:
dataset_path = '/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/splits/dataset.csv'
df_dataset = pd.read_csv(dataset_path)
print("----- Dataset CSV -----")
print(df_dataset.head())
print(type(df_dataset), getattr(df_dataset, "shape", None))
print(df_dataset.columns)

prototype_path = '/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/splits/prototype.csv'
df_prototype = pd.read_csv(prototype_path)
print("----- Prototype CSV -----")
print(df_prototype.head())
print(type(df_prototype), getattr(df_prototype, "shape", None))
print(df_prototype.columns)

test_path = '/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/splits/test.csv'
df_test = pd.read_csv(test_path)
print("----- Test CSV -----")
print(df_test.head())
print(type(df_test), getattr(df_test, "shape", None))
print(df_test.columns)


----- Dataset CSV -----
             uid             gloss  duration         category
0  --7-rCNOiK0_1            branch       6.0  flora and fauna
1    -02y2VaVBmw  divorce petition      11.0            legal
2    -03Dju3yPHE   air conditioner       5.0          machine
3    -0Oo1kPonYQ            stapoo       6.0            sport
4  -0XujLX2_9I_2       application       6.0           action
<class 'pandas.core.frame.DataFrame'> (7050, 4)
Index(['uid', 'gloss', 'duration', 'category'], dtype='object')
----- Prototype CSV -----
             uid             gloss  duration         category
0  --7-rCNOiK0_1            branch       6.0  flora and fauna
1    -02y2VaVBmw  divorce petition      11.0            legal
2    -03Dju3yPHE   air conditioner       5.0          machine
3    -0Oo1kPonYQ            stapoo       6.0            sport
4  -0XujLX2_9I_2       application       6.0           action
<class 'pandas.core.frame.DataFrame'> (4765, 4)
Index(['uid', 'gloss', 'duration', 'category']

In [20]:
df_combined = pd.concat([df_prototype['uid'].to_frame(), df_test['uid'].to_frame()], ignore_index=True)

In [21]:
print("----- Combined UIDs -----")
print(df_combined.head())

----- Combined UIDs -----
             uid
0  --7-rCNOiK0_1
1    -02y2VaVBmw
2    -03Dju3yPHE
3    -0Oo1kPonYQ
4  -0XujLX2_9I_2


In [22]:
(features_df["id"].values == df_combined["uid"].values).all()


np.False_

In [23]:
feature_set = set(features_df["id"].values)
combined_set = set(df_combined["uid"].values)

missing_ids = combined_set - feature_set
print(f"Missing IDs in features: {len(missing_ids)}")

Missing IDs in features: 0


In [24]:
df_prototype["gloss"].nunique()

4764

In [26]:
print(df_prototype["gloss"].nunique(), len(df_prototype))
print(df_test["gloss"].nunique(), len(df_test))

4764 4765
1363 2285


In [27]:
print(features_df.shape)

(7050, 2)


In [30]:
print(type(features_df.loc[0, "I3D_features"]))
print(len(features_df.loc[0, "I3D_features"]))
print(np.array(features_df.loc[0, "I3D_features"]).shape)
print(np.array(features_df.loc[0, "I3D_features"]).reshape(-1).shape)


<class 'numpy.ndarray'>
1
(1, 1024, 11, 1, 1)
(11264,)


In [43]:
for gloss in df_prototype["gloss"].unique():
    video_list = df_prototype[df_prototype["gloss"] == gloss]["uid"].tolist()
    #print(f"Gloss: {gloss}, Video UIDs: {video_list}")
print(video_list)

['zxwuZPkYjrQ']


In [105]:
prototype_vectors = []
prototype_glosses = []
df_prototype = df_prototype.dropna(subset=["gloss", "uid"]).copy()
df_prototype["gloss"] = df_prototype["gloss"].astype(str).str.strip()
df_prototype["uid"] = df_prototype["uid"].astype(str).str.strip()
for gloss in df_prototype["gloss"].unique():
    video_list = df_prototype[df_prototype["gloss"] == gloss]["uid"].tolist()
    #print(f"Gloss: {gloss}, Video UIDs: {video_list}")

    per_video_vectors = []
    
    if len(video_list) == 0:
        continue

    for video in video_list:
        # Step 2.1: extract and squeeze
        tensor = features_df.loc[
            features_df["id"] == video, "I3D_features"
        ].iloc[0].squeeze()          # (1024, 11)

        # Step 2.2: temporal average
        video_vector = tensor.mean(axis=1)  # (1024,)

        # Step 2.3: L2 normalize
        norm = np.linalg.norm(video_vector)
        if norm > 1e-8:
            video_vector = video_vector / norm

        per_video_vectors.append(video_vector)

    # Aggregate prototypes (handles 1 or 2 videos automatically)
    prototype_vector = np.mean(per_video_vectors, axis=0)

    # Final normalization
    norm = np.linalg.norm(prototype_vector)
    if norm > 1e-8:
        prototype_vector = prototype_vector/norm

    prototype_vectors.append(prototype_vector)
    prototype_glosses.append(gloss)


In [106]:
test_vectors = []
test_glosses = []
df_test["gloss"] = df_test["gloss"].str.strip()
for gloss in df_test["gloss"].unique():
    video_list = df_test[df_test["gloss"] == gloss]["uid"].tolist()
    #print(f"Gloss: {gloss}, Video UIDs: {video_list}")

    per_video_vectors = []
    

    for video in video_list:
        # Step 2.1: extract and squeeze
        tensor = features_df.loc[
            features_df["id"] == video, "I3D_features"
        ].iloc[0].squeeze()          # (1024, 11)

        # Step 2.2: temporal average
        video_vector = tensor.mean(axis=1)  # (1024,)

        # Step 2.3: L2 normalize
        norm = np.linalg.norm(video_vector)
        if norm > 1e-8:
            video_vector = video_vector / norm

        per_video_vectors.append(video_vector)

    # Aggregate prototypes (handles 1 or 2 videos automatically)
    test_vector = np.mean(per_video_vectors, axis=0)

    # Final normalization
    norm = np.linalg.norm(test_vector)
    if norm > 1e-8:
        test_vector = test_vector/norm

    test_vectors.append(test_vector)
    test_glosses.append(gloss)


In [92]:
def normalize_gloss(g):
    return " ".join(g.upper().strip().split())


In [107]:
def normalize_gloss(g):
    return " ".join(g.upper().strip().split())

top1 = top5 = top10 = 0
prototype_matrix = np.stack(prototype_vectors, axis=0)  # (P, 1024)
test_matrix = np.stack(test_vectors, axis=0)            # (T, 1024)
prototype_glosses = [g.strip().upper() for g in prototype_glosses]
test_glosses = [g.strip().upper() for g in test_glosses]
prototype_glosses = [normalize_gloss(g) for g in prototype_glosses]
test_glosses = [normalize_gloss(g) for g in test_glosses]

i = 0
true = test_glosses[i]

sims = prototype_matrix @ test_matrix[i]
idx = np.argsort(sims)[::-1][:20]
ranked = [prototype_glosses[j] for j in idx]

print(true, ranked)


for v_test, true_gloss in zip(test_matrix, test_glosses):

    sims = prototype_matrix @ v_test
    ranked_indices = np.argsort(sims)[::-1]
    ranked_glosses = [prototype_glosses[i] for i in ranked_indices]

    if ranked_glosses[0] == true_gloss:
        top1 += 1
    if true_gloss in ranked_glosses[:5]:
        top5 += 1
    if true_gloss in ranked_glosses[:10]:
        top10 += 1

top1_acc  = (top1  / len(test_glosses))*100
top5_acc  = (top5  / len(test_glosses))*100
top10_acc = (top10 / len(test_glosses))*100

print(top1_acc, top5_acc, top10_acc)



SEVERALLY ['RISK', 'FRIEND', 'SANCTION', 'GROUP', 'SIGNATURE', 'SIGNATORY', 'GOOD', 'RESTORE', 'STOCK', 'RESOLUTION', 'REVISED', 'RETAIN', 'SAFE GUARD', 'RUNNING ACCOUNT', 'SEIZURE', 'SETTLE', 'FATHER-IN-LAW', 'ADD', 'LOCK', 'TRACK BALL']
21.716801173881144 25.825385179750548 28.173147468818783


In [125]:
print(len(prototype_vectors))
print(prototype_vectors[:1])

4764
[array([0.00728226, 0.02289256, 0.00882173, ..., 0.02663012, 0.03285389,
       0.00294585], shape=(1024,), dtype=float32)]


In [116]:
print(df_test.columns)


Index(['uid', 'gloss', 'duration', 'category'], dtype='object')


#### Self-retrieval check (must pass): for each prototype, query it against the prototype bank; rank of itself should be 1 almost always.

In [104]:
def normalize_gloss(g):
    return " ".join(g.upper().strip().split())

top1 = top5 = top10 = 0
prototype_matrix = np.stack(prototype_vectors, axis=0)  # (P, 1024)
test_matrix = np.stack(test_vectors, axis=0)            # (T, 1024)
prototype_glosses = [g.strip().upper() for g in prototype_glosses]
test_glosses = [g.strip().upper() for g in test_glosses]
prototype_glosses = [normalize_gloss(g) for g in prototype_glosses]
test_glosses = [normalize_gloss(g) for g in test_glosses]

i = 0
true = test_glosses[i]

sims = prototype_matrix @ test_matrix[i]
idx = np.argsort(sims)[::-1][:20]
ranked = [prototype_glosses[j] for j in idx]

print(true, ranked)


for p_test, true_gloss in zip(test_matrix, test_glosses):

    sims = test_matrix @ p_test
    ranked_indices = np.argsort(sims)[::-1]
    ranked_glosses = [test_glosses[i] for i in ranked_indices]

    if ranked_glosses[0] == true_gloss:
        top1 += 1
    if true_gloss in ranked_glosses[:5]:
        top5 += 1
    if true_gloss in ranked_glosses[:10]:
        top10 += 1

top1_acc  = (top1  / len(test_glosses))*100
top5_acc  = (top5  / len(test_glosses))*100
top10_acc = (top10 / len(test_glosses))*100

print(top1_acc, top5_acc, top10_acc)



SEVERALLY ['GOOD MORNING', 'GOOD NIGHT', 'LAST MONTH', 'NEXT', 'CIRCLE', 'OFFICIAL', 'TRACK BALL', 'ONLINE', 'GUAVA', 'SUPER COMPUTER', 'DROPPED ARMHOLE SLEEVE', 'MEDITATE', 'RESUME', 'FIR', 'PREFER', 'LOG OFF', 'ENERGY METER', 'XBOX', 'PASTE', 'INNOVATE']
85.5465884079237 100.0 100.0


#### Step 2 — Make similarity truly “cosine” (and verify the effect)


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import top_k_accuracy_score
from sklearn.preprocessing import train_test_split
from sklearn.preprocessing import LabelEncoder


In [ ]:

le = LabelEncoder()
df_prototype["encoded_category"] = le.fit_transform(df_prototype["category"].values)



                      gloss category
4               application   action
7                   squeeze   action
10                   defend   action
16                  testify   action
27                    think   action
...                     ...      ...
4718  ogle from head to toe   action
4720          prompt action   action
4745               approval   action
4755              take over   action
4762               postpone   action

[447 rows x 2 columns]


In [ ]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]
features = prototype_matrix
dataset = CustomDataset(features, labels)


In [160]:
from collections import Counter

shapes = [np.asarray(x).shape for x in df_training["I3D_features"]]
Counter(shapes)


Counter({(1, 1024, 9, 1, 1): 573,
         (1, 1024, 11, 1, 1): 495,
         (1, 1024, 12, 1, 1): 484,
         (1, 1024, 14, 1, 1): 435,
         (1, 1024, 10, 1, 1): 373,
         (1, 1024, 7, 1, 1): 314,
         (1, 1024, 16, 1, 1): 282,
         (1, 1024, 13, 1, 1): 213,
         (1, 1024, 8, 1, 1): 150,
         (1, 1024, 18, 1, 1): 149,
         (1, 1024, 15, 1, 1): 135,
         (1, 1024, 20, 1, 1): 112,
         (1, 1024, 17, 1, 1): 100,
         (1, 1024, 22, 1, 1): 77,
         (1, 1024, 21, 1, 1): 50,
         (1, 1024, 26, 1, 1): 49,
         (1, 1024, 31, 1, 1): 43,
         (1, 1024, 19, 1, 1): 43,
         (1, 1024, 25, 1, 1): 43,
         (1, 1024, 29, 1, 1): 42,
         (1, 1024, 41, 1, 1): 40,
         (1, 1024, 6, 1, 1): 38,
         (1, 1024, 28, 1, 1): 38,
         (1, 1024, 44, 1, 1): 38,
         (1, 1024, 43, 1, 1): 37,
         (1, 1024, 24, 1, 1): 37,
         (1, 1024, 35, 1, 1): 36,
         (1, 1024, 40, 1, 1): 35,
         (1, 1024, 32, 1, 1): 31,
     